# RLHF with Hugging Face TRL

This notebook runs the repository-owned HelpSteer3 preprocessing, TRL SFT, reward modeling, TRL PPO, policy-suite evaluation, and qualitative audit. Use `smoke` for a small end-to-end check; use `full` for the A100 run.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/rlhf.git"
REPO_DIR = Path("/content/rlhf")
RUN_PROFILE = "full"  # "smoke" or "full"
RUN_TAG = "qwen25_05b_helpsteer3_trl_a100_full"
RUN_FULL_EVAL = True  # Full profile evaluates all 2,017 validation prompts.
DRIVE_SYNC_ROOT = None  # Example: "/content/drive/MyDrive/rlhf_runs"

assert RUN_PROFILE in {"smoke", "full"}

In [ ]:
import os
import subprocess
import sys

def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "configs" / "trl").exists()

def find_repo_root() -> Path | None:
    for start in (Path.cwd(), REPO_DIR):
        for candidate in (start, *start.parents):
            if is_repo_root(candidate):
                return candidate
    return None

root = find_repo_root()
if root is None:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    root = REPO_DIR
os.chdir(root)
print(f"Repository root: {root}")

if DRIVE_SYNC_ROOT and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-rlhf.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

In [ ]:
import json
import shlex
import shutil
import accelerate
import datasets
import peft
import torch
import transformers
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
assert tuple(map(int, torch.__version__.split("+", 1)[0].split(".")[:2])) >= (2, 6)
assert trl.__version__ == "1.6.0"
assert transformers.__version__.startswith("4.")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before training.")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("GPU memory (GiB):", round(props.total_memory / 2**30, 1))
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
LOCAL_RUN_ROOT = Path("/content/rlhf_runs") / RUN_TAG / RUN_PROFILE
CACHE_DIR = LOCAL_RUN_ROOT / "data"
SFT_DIR = LOCAL_RUN_ROOT / "sft"
REWARD_DIR = LOCAL_RUN_ROOT / "reward"
PPO_DIR = LOCAL_RUN_ROOT / "ppo"
EVAL_DIR = LOCAL_RUN_ROOT / "eval"
for path in (CACHE_DIR, SFT_DIR, REWARD_DIR, PPO_DIR, EVAL_DIR):
    path.mkdir(parents=True, exist_ok=True)

DRIVE_RUN_ROOT = Path(DRIVE_SYNC_ROOT) / RUN_TAG / RUN_PROFILE if DRIVE_SYNC_ROOT else None

def encode_override(key, value):
    if isinstance(value, (str, bool, int, float)) or value is None:
        rendered = json.dumps(value)
    else:
        rendered = json.dumps(value, separators=(",", ":"))
    return f"{key}={rendered}"

def run_script(script, config, overrides=None, extra_args=None):
    command = [sys.executable, script, "--config", config]
    for key, value in (overrides or {}).items():
        command.extend(["--set", encode_override(key, value)])
    command.extend(extra_args or [])
    print("$", shlex.join(command))
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    process = subprocess.Popen(
        command,
        cwd=root,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line)
        tail = tail[-120:]
    return_code = process.wait()
    if return_code:
        detail = "".join(tail)
        raise RuntimeError(
            f"Command failed with exit status {return_code}: {shlex.join(command)}\n\n"
            f"Last child-process output:\n{detail}"
        )

def sync_overrides(stage):
    if DRIVE_RUN_ROOT is None:
        return {}
    return {
        "train.checkpoint_sync_dir": str(DRIVE_RUN_ROOT / stage / "checkpoints"),
        "train.final_sync_dir": str(DRIVE_RUN_ROOT / stage),
    }

SMOKE = RUN_PROFILE == "smoke"
print("Run root:", LOCAL_RUN_ROOT)

## 1. Prepare response-safe datasets

In [ ]:
prepare_overrides = {
    "data.cache_dir": str(CACHE_DIR),
}
if SMOKE:
    prepare_overrides.update({
        "data.max_train_samples": 256,
        "data.max_validation_samples": 64,
        "data.max_total_length": 1024,
        "data.max_prompt_length": 768,
    })
run_script(
    "scripts/rlhf_trl_prepare_data.py",
    "configs/trl/qwen25_05b_helpsteer3_sft.yaml",
    prepare_overrides,
)

## 2. Supervised fine-tuning

In [ ]:
sft_overrides = {
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(SFT_DIR),
    **sync_overrides("sft"),
}
if SMOKE:
    sft_overrides.update({
        "data.max_train_samples": 128,
        "data.max_eval_samples": 32,
        "train.max_steps": 8,
        "train.per_device_train_batch_size": 2,
        "train.gradient_accumulation_steps": 2,
        "train.eval_steps": 4,
        "train.save_steps": 4,
    })
run_script(
    "scripts/rlhf_trl_doctor.py",
    "configs/trl/qwen25_05b_helpsteer3_sft.yaml",
    sft_overrides,
    extra_args=["--stage", "sft"],
)
run_script("scripts/rlhf_trl_train_sft.py", "configs/trl/qwen25_05b_helpsteer3_sft.yaml", sft_overrides)

## 3. Reward model

In [ ]:
reward_overrides = {
    "model.sft_model_path": str(SFT_DIR / "final_merged_model"),
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(REWARD_DIR),
    **sync_overrides("reward"),
}
if SMOKE:
    reward_overrides.update({
        "data.max_train_samples": 128,
        "data.max_eval_samples": 32,
        "data.max_center_samples": 64,
        "train.max_steps": 8,
        "train.per_device_train_batch_size": 2,
        "train.gradient_accumulation_steps": 2,
        "train.eval_steps": 4,
        "train.save_steps": 4,
        "train.audit_batch_size": 8,
    })
run_script(
    "scripts/rlhf_trl_doctor.py",
    "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
    reward_overrides,
    extra_args=["--stage", "reward"],
)
run_script(
    "scripts/rlhf_trl_train_reward_model.py",
    "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
    reward_overrides,
)

## 4. PPO alignment

In [ ]:
ppo_overrides = {
    "model.policy_model_path": str(SFT_DIR / "final_merged_model"),
    "model.reference_model_path": str(SFT_DIR / "final_merged_model"),
    "model.reward_model_path": str(REWARD_DIR / "final_merged_model"),
    "model.reward_center_path": str(REWARD_DIR / "reward_center.json"),
    "model.value_model_path": str(REWARD_DIR / "final_merged_model"),
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(PPO_DIR),
    **sync_overrides("ppo"),
}
if SMOKE:
    ppo_overrides.update({
        "data.max_train_samples": 64,
        "data.max_eval_samples": 16,
        "ppo.total_episodes": 32,
        "ppo.response_length": 128,
        "ppo.local_rollout_forward_batch_size": 2,
        "ppo.num_sample_generations": 2,
        "train.per_device_train_batch_size": 1,
        "train.gradient_accumulation_steps": 8,
        "train.save_steps": 2,
    })
run_script(
    "scripts/rlhf_trl_doctor.py",
    "configs/trl/qwen25_05b_helpsteer3_ppo.yaml",
    ppo_overrides,
    extra_args=["--stage", "ppo"],
)
run_script("scripts/rlhf_trl_train_ppo.py", "configs/trl/qwen25_05b_helpsteer3_ppo.yaml", ppo_overrides)

## 5. Base / SFT / PPO evaluation

In [ ]:
eval_overrides = {
    "model.tokenizer_path": str(SFT_DIR / "final_merged_model"),
    "reward_model.checkpoint_dir": str(REWARD_DIR / "final_merged_model"),
    "reward_model.reward_center_path": str(REWARD_DIR / "reward_center.json"),
    "policies.0.label": "base",
    "policies.0.checkpoint_dir": None,
    "policies.1.checkpoint_dir": str(SFT_DIR / "final_merged_model"),
    "policies.2.checkpoint_dir": str(PPO_DIR / "final_merged_policy"),
    "eval.output_dir": str(EVAL_DIR),
    "eval.num_prompts": 32 if SMOKE else ("all" if RUN_FULL_EVAL else 128),
}
if SMOKE:
    eval_overrides.update({
        "eval.batch_size": 4,
        "generation.max_prompt_length": 768,
        "generation.max_new_tokens": 128,
    })
run_script(
    "scripts/rlhf_evaluate_policy_suite.py",
    "configs/trl/qwen25_05b_helpsteer3_eval_suite.yaml",
    eval_overrides,
)

## 6. Qualitative audit

In [ ]:
audit_command = [
    sys.executable,
    "scripts/rlhf_audit_policy_suite.py",
    "--eval-dir", str(EVAL_DIR),
    "--base-label", "base",
    "--sft-label", "sft_trl",
    "--ppo-label", "ppo_trl",
]
print("$", shlex.join(audit_command))
subprocess.run(audit_command, cwd=root, check=True)
print("Audit:", EVAL_DIR / "qualitative_audit_auto.md")

if DRIVE_RUN_ROOT is not None:
    destination = DRIVE_RUN_ROOT / "eval"
    shutil.copytree(EVAL_DIR, destination, dirs_exist_ok=True)
    print("Evaluation copied to:", destination)

## Before a full run

Read the smoke summaries and generated responses before launching the full profile. Continue only if reward-model accuracy and calibration are credible, PPO metrics remain finite with bounded KL, responses terminate sensibly, and repetition does not sharply increase. The full profile uses the complete training set, 100,000 PPO episodes, 1024-token PPO rollouts, and the full 2,017-prompt, 1024-token policy-suite evaluation.